# Hello World of Optimization! 
Let's Build an EV Charging Network
## Our task...
We are in charge of deploying EV charging stations for a municipal parking authority.
- Management would like to install these new charging stations using the available resources from our current budget and infrastructure capacity.
- Right now, we are asked to **maximize the total number of vehicles we can serve per day** with our charging network.
- The charging station types we can install are Level 1 (120V), Level 2 (240V), DC Fast 50kW, DC Fast 150kW, and DC Ultra-Fast 350kW.
- Each type of charging station requires different amounts of several limited resources.
- We are constrained by our available **budget, electrical capacity, physical space, and equipment units**.

## Initial questions
- What are our decisions?
- What are our constraints? 
- What is the objective?
- What data do we need right now?

## The Model
A lot of programming (in Python) is *imperative* -- just providing sequential instructions to complete. But mathematical optimization (aka math programming) is *declarative*. The math programming model does not tell the Gurobi solver what to do specifically. Instead, the model tells the Gurobi solver what the solution must look like. Gurobi then finds the solution in its own way.

So math programming always starts with the creation of a new model. We then add to it the *declarations* about the final solution. 

Math optimization models take two forms:
- The **formulation**: An algebraic representation of the model (what we saw in our introduction!)
- The **code**: Writing the formulation in syntax to some software package.

In [ ]:
#%pip install gurobipy

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [ ]:
model = gp.Model("EV Charging Network")

### The Variables

The model needs something to receive the solution values that the solver finds. It declares *variables* to hold those solution values. The mathematical optimization model never explicitly sets these values. It just describes them and makes rules about the values they can hold. This model declares a variable for each type of charging station.

Let $x_i$ be the number of charging stations installed of type $i \in \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$.

First, let's create data for each charging station type and its characteristics.

In [ ]:
chargers = ["Level1", "Level2", "Fast50", "Fast150", "Ultra350"]
data = pd.Series([3, 6, 32, 48, 80], 
                  index=chargers, 
                  name='vehicles_per_day')
data

#### Anyone know another way to hardcode this another way?... using a `gurobipy` object?
**Multidict** function splits one dictionary into multiple, allowing for easier access to data while model building

In [ ]:
chargers, vehicles_per_day = gp.multidict(
    {
        "Level1":     3,
        "Level2":     6,
        "Fast50":    32,
        "Fast150":   48,
        "Ultra350":  80
    }
)

In [ ]:
print(chargers)

In [ ]:
print(vehicles_per_day)

Next, variables are added to the model -- one for each charger type. The main types of decision variables are
- CONTINUOUS
- INTEGER
- BINARY

But we'll explore more types later.

#### What type of decision variable should we use in this case?
- Our decisions are the *number of charging stations* we should install for each type

In [ ]:
x = model.addVars(chargers, vtype = GRB.INTEGER, name = "chargers")
model.update()
x

### The Objective Function:

The math model must describe an objective also using algebra. When the model is solved, the value of the objective function will be the maximum or minimum possible while following the rules described in the constraints.

This model will maximize the total number of vehicles served per day. The objective function multiplies the quantity of each charger type installed times its daily vehicle throughput for all charger types.
For example, the total vehicles served by Level 2 chargers is $8*x_{Level2}$.

So the total vehicles served per day is

\begin{equation*}
3*x_{Level1} + 6*x_{Level2} + 32*x_{Fast50} + ... + 80*x_{Ultra350}
\end{equation*}

In [ ]:
### Option 1 - Written term by term:
model.setObjective(x["Level1"] * vehicles_per_day["Level1"] + 
                   x["Level2"] * vehicles_per_day["Level2"] + 
                   x["Fast50"] * vehicles_per_day["Fast50"] + 
                   x["Fast150"] * vehicles_per_day["Fast150"] + 
                   x["Ultra350"] * vehicles_per_day["Ultra350"], 
                    sense=GRB.MAXIMIZE)

Let $v_i$ be the vehicles served per day by charger type $i$ and use a bit more math notation:
\begin{equation*}
  \text{Maximize} \space \sum_i v_i*x_i
\end{equation*}

In [ ]:
### Option 2 - Here is another way to write the same thing:
model.setObjective(gp.quicksum(x[i] * vehicles_per_day[i] for i in chargers), sense=GRB.MAXIMIZE)

### Option 3 - The prod function is a handy function provided by the Gurobi API.
model.setObjective(x.prod(vehicles_per_day), sense=GRB.MAXIMIZE)

### The Constraints:

This parking authority has limited resources available to deploy the charging stations. The model must make sure to only install chargers for which the resources are available.

Here is a list of resources and how much is available:

In [ ]:
resources, available = gp.multidict(
    {
        "budget":     500000,  # dollars
        "power":        2000,  # kW
        "space":         600,  # square meters
        "equipment":     100,  # units
    }
)

In [ ]:
print(resources)
print(available)

In [ ]:
requirements = {
    ("Level1",   "budget"):    500, ("Level1",   "power"):   1.4, ("Level1",   "space"):  12, ("Level1",   "equipment"):  0,
    ("Level2",   "budget"):   3000, ("Level2",   "power"):   7.2, ("Level2",   "space"):  12, ("Level2",   "equipment"):  1,
    ("Fast50",   "budget"):  20000, ("Fast50",   "power"):  50.0, ("Fast50",   "space"):  18, ("Fast50",   "equipment"):  2,
    ("Fast150",  "budget"):  60000, ("Fast150",  "power"): 150.0, ("Fast150",  "space"):  20, ("Fast150",  "equipment"):  4,
    ("Ultra350", "budget"): 120000, ("Ultra350", "power"): 350.0, ("Ultra350", "space"):  25, ("Ultra350", "equipment"):  6,
}

In [ ]:
requirements['Level2', 'power']

Let's introduce **constraints** to make sure we don't use more resources than we have available. Constraints are where the *rules* acting on decision variables are declared.

For example, the amount of power used by all chargers must be less than or equal to the total power capacity available.

\begin{equation*}
\text{total power used} \le 2000
\end{equation*}

Let's write an expression for the total power used using our decision variable, $x_{power}$.
\begin{align*}
\text{total power used} = \space& power_{Level1}* x_{Level1} + power_{Level2}*x_{Level2} + ... + power_{Ultra350}*x_{Ultra350} \\
& power_{Level1}* x_{Level1} + power_{Level2}*x_{Level2} + ... + power_{Ultra350}*x_{Ultra350} \le 2000
\end{align*}

Given we stored this information in the `requirements` dictionary, we let $r_{i,j}$ be the amount of *resource* $j$ required by *charger type* $i$.
\begin{equation*}
  r_{Level1, power} * x_{Level1} + r_{Level2, power}*x_{Level2} + ... + r_{Ultra350, power}*x_{Ultra350} \le 2000
\end{equation*}

In [ ]:
### Option 1 - A very explicit way to write these constraints:

# total budget <= budget available
model.addConstr(x["Level1"] * requirements["Level1", "budget"] + x["Level2"] * requirements["Level2", "budget"]
                + x["Fast50"] * requirements["Fast50", "budget"] + x["Fast150"] * requirements["Fast150", "budget"]
                + x["Ultra350"] * requirements["Ultra350", "budget"] <= available["budget"], "budget limit")

# total power <= power available
model.addConstr(x["Level1"] * requirements["Level1", "power"] + x["Level2"] * requirements["Level2", "power"]
                + x["Fast50"] * requirements["Fast50", "power"] + x["Fast150"] * requirements["Fast150", "power"]
                + x["Ultra350"] * requirements["Ultra350", "power"] <= available["power"], "power limit")

# total space <= space available
model.addConstr(x["Level1"] * requirements["Level1", "space"] + x["Level2"] * requirements["Level2", "space"]
                + x["Fast50"] * requirements["Fast50", "space"] + x["Fast150"] * requirements["Fast150", "space"]
                + x["Ultra350"] * requirements["Ultra350", "space"] <= available["space"], "space limit")

# total equipment <= equipment available
model.addConstr(x["Level1"] * requirements["Level1", "equipment"] + x["Level2"] * requirements["Level2", "equipment"]
                + x["Fast50"] * requirements["Fast50", "equipment"] + x["Fast150"] * requirements["Fast150", "equipment"]
                + x["Ultra350"] * requirements["Ultra350", "equipment"] <= available["equipment"], "equipment limit")

We can also generalize the *quantity* of each resource with $q_j$. Then using a bit more mathematical notation:
\begin{equation*}
  \sum_{i}r_{i, j} * x_{i} \le q_j, \space \text{for all} \space j \space \text{in} \space \{\text{budget, power, space, equipment}\}
\end{equation*}

Note: A short way to write "for all" is using the symbol $\forall$. Also, $\in$ means "in", or "an element of." We saw this notation in the intro.

Where the following can be found using requirements[('Level2', 'power')]
\begin{equation*} r_{Level2, power} \end{equation*}

We can change the code so it looks like this condensed notation and loop through each resource using `quicksum`:

In [ ]:
### Option 2 - Using quicksum
for resource in resources:
    model.addConstr(gp.quicksum(x[i] * requirements[i, resource] for i in chargers) <= available[resource], name="resource usage")

A little more compact, and makes it easier to store the constraints as an object:

In [ ]:
### Option 3 - Using quicksum and storing constraint as an object
resource_constraints = model.addConstrs((gp.quicksum(x[i] * requirements[i, j] for i in chargers) <= available[j] for j in resources), name="resource usage")

### The Solution:
It's as simple as one line of code to run the optimization, then we query the decision variables for their values (assuming the optimization completed successfully)

In [ ]:
model.optimize()

In [ ]:
# use `VarName` and `X` to get the variables name and value, respectively:
for v in model.getVars():
    print(f"{v.VarName}: {v.X}")

Do you think another (and simpler) approach may work for this specific version of the problem?

## Congratulations!
You have just ran an optimization model!

# More Realistic Problems
Where we left off, a greedy approach would have worked to solve that model. But problems typically have much more complicated business rules to follow -- which highlights the strengths of mathematical optimization.

## Let's Create a function to make solving and reporting a bit more straightforward

In [ ]:
def solve_and_print_solution(model, x, chargers):
    # Optimize model
    model.setParam("outputflag", 0)
    model.optimize()
    
    # Check the model status
    if model.status == GRB.OPTIMAL:
        print("Optimal solution found")
        # Print the results
        for v in model.getVars():
            print(f"{v.VarName}: {v.X}")
        print(f"Objective value: {model.objVal}")
    else:
        print(f"Model status: {model.status}")
        sys.exit(1)

## Pick up where we left off
When setting up the problem the first time we used `multidict` to create parts of our model. This time, let's read data in from a csv.

If you are on colab, use this cell.

In [ ]:
### This time, we read in data from csv
path = 'https://raw.githubusercontent.com/https://github.com/caroweinberg/ODSC_AI_East_2026/refs/heads/main'
available = pd.read_csv(path+'data_files/resource_available.csv', index_col=['resource']).squeeze()
chargers_data = pd.read_csv(path+'data_files/chargers_data.csv', index_col=['chargers']).squeeze()
requirements = pd.read_csv(path+'data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()

resources = available.index.to_list()
chargers = chargers_data.index.to_list()

If you downloaded the files locally, run this cell:

In [ ]:
### This time, we read in data from csv
available = pd.read_csv('data_files/resource_available.csv', index_col=['resource']).squeeze()
chargers_data = pd.read_csv('data_files/chargers_data.csv', index_col=['chargers']).squeeze()
requirements = pd.read_csv('data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()

resources = available.index.to_list()
chargers = chargers_data.index.to_list()

## Original model

As a reminder:
- $x_i$ is the number of charging stations to install of type $i \in \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$.
- $v_i$ is the vehicles served per day by each charger type.
- $q_j$ is the amount of resource $j \in \{\text{budget, power, space, equipment}\}$ we have available.
- $r_{i,j}$ is the amount of **resource** $j$ required to install **charger type** $i$.

\begin{align*}
&\text{Maximize} \space \sum_i v_i*x_i \\
&\text{subject to:} \\
&\quad\sum_{i}r_{i, j} * x_{i} \le q_j, \space \text{for} \space j \in \{\text{budget, power, space, equipment}\} \\
&\quad x_i \ge 0, \forall i \in \text{Chargers}
\end{align*}

In [ ]:
### Create a new model
model = gp.Model("EV Charging Network")

### Decision variables
x = model.addVars(chargers, vtype=GRB.INTEGER, name="chargers")

### Resource constraints
resource_usage = model.addConstrs((gp.quicksum(x[i] * requirements[i, j] for i in chargers) <= available[j] for j in resources), name="resource_usage")

### Objective function
model.setObjective(gp.quicksum(x[i] * chargers_data.vehicles_per_day[i] for i in chargers), sense=GRB.MAXIMIZE)

### Optimize
solve_and_print_solution(model, x, chargers)

## Decision expressions
Sometimes it is helpful to store quantities of interest to help make code easier to read, reduce clutter, or get key values quickly.

Suppose we have grouped charger types into two categories: standard and fast. Fast chargers are Fast50, Fast150, and Ultra350, with the rest being standard.

Code the expressions below containing decision variables.

Let's define some sets.
- $I = \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$
- $F = \{\text{Fast50, Fast150, Ultra350}\}$
- $S = I - F$

\begin{align*}
\text{total chargers} &= \sum_{i \in I} x_i, \\
\text{vehicles served by all chargers} &= \sum_{i \in I} v_i*x_i \\
\text{vehicles served by fast chargers} &= \sum_{f \in F} v_f * x_f \\
\text{vehicles served by standard chargers} &= \sum_{s \in S} v_s*x_s \\
\end{align*}

In [ ]:
fast_chargers = ['Fast50', 'Fast150', 'Ultra350']
standard_chargers = [i for i in chargers if i not in fast_chargers]

### While we use `quicksum` here, there are more compact ways to write these.
total_chargers = gp.quicksum(x[i] for i in chargers)
vehicles_all_chargers = gp.quicksum(chargers_data.vehicles_per_day[i] * x[i] for i in chargers)
vehicles_fast_chargers = gp.quicksum(chargers_data.vehicles_per_day[i] * x[i] for i in fast_chargers)
vehicles_standard_chargers = gp.quicksum(chargers_data.vehicles_per_day[i] * x[i] for i in standard_chargers)

# Update model object to apply changes
model.update()

In [ ]:
### Test your expressions here
#vehicles_standard_chargers.getValue() + vehicles_fast_chargers.getValue()
total_chargers.getValue()

## Changing our model

### New objective function
Instead of maximizing the vehicles served, we are asked to maximize the total number of charging stations installed. Write out and code this new objective

\begin{equation*}
\text{Maximize} \space \sum_{i \in I} x_i
\end{equation*}

In [ ]:
### Set the objective to maximize the number of charging stations
model.setObjective(total_chargers, GRB.MAXIMIZE)

# When you call optimize, the model is updated automatically, no need to call model.update()
solve_and_print_solution(model, x, chargers)

Note that we didn't have to do anything else other than re-run `setObjective`.

### Service Balance Between Standard and Fast Chargers

Let's add a constraint and see what happens!

#### Formulate and code the following constraint
- The vehicles served by fast chargers cannot exceed 60% of the vehicles served by standard chargers.

**Question**: Will this constraint change our current solution where we're maximizing the number of stations?

\begin{equation*}
\sum_{f \in F} v_f*x_f \le 0.6*\sum_{s \in S} v_s*x_s
\end{equation*}

In [ ]:
### The vehicles served by fast chargers cannot exceed 60% of vehicles served by standard chargers
service_balance = model.addConstr(vehicles_fast_chargers <= 0.6 * vehicles_standard_chargers, name='service_balance')

solve_and_print_solution(model, x, chargers)

**Discussion**: The constraint didn't change anything! Why?
- Our current solution has 0 fast chargers (only Level 1s to maximize station count)
- The constraint is: $0 \le 0.6 \times (\text{positive number})$
- This is always satisfied - it's a **non-binding constraint**

**Key Learning**: Not all constraints are active in every solution! Some constraints only matter when certain decisions are made.

**To see this constraint in action**, we need a scenario where fast chargers might actually be installed. Let's switch back to maximizing vehicles served:

In [ ]:
### Change objective back to maximize vehicles served
model.setObjective(vehicles_all_chargers, GRB.MAXIMIZE)
solve_and_print_solution(model, x, chargers)

**Now the constraint matters!** When maximizing vehicles served, we want fast chargers, but the service balance constraint limits how many we can install.

We stored each constraint as an object, which makes it easier to interact with these later. Now, let's remove the `service_balance` constraint.

In [ ]:
model.remove(service_balance)

### Proportion constraints

#### Formulate and code the following constraints
- Each of the standard chargers must be at least 15% of the total chargers installed and each of the fast chargers cannot be more than 35%.
- The Level 2 charger is our workhorse; make sure that it has the maximum number among all chargers.

\begin{align*}
x_{s} \ge& 0.15*\sum_{i \in I} x_i, \forall s\in S \\
x_{f} \le& 0.35*\sum_{i \in I} x_i, \forall f\in F \\
\end{align*}

In [ ]:
standard_charger_lb = model.addConstrs((x[s] >= 0.15 * total_chargers for s in standard_chargers), name='standard_charger_lb')
fast_charger_ub = model.addConstrs((x[f] <= 0.35 * total_chargers for f in fast_chargers), name='fast_charger_ub')

solve_and_print_solution(model, x, chargers)

\begin{align*}
x_{Level2} \ge x_{i}, \forall i \in I-\{Level2\}.
\end{align*}

In [ ]:
### Use this set since this constraint doesn't need to apply to Level2
I_minusLevel2 = [i for i in chargers if i != 'Level2']

max_level2 = model.addConstrs((x['Level2'] >= x[i] for i in I_minusLevel2), name='max_level2')

solve_and_print_solution(model, x, chargers)

### Let's check in on our model and remove some constraints
As you code a model, you may want to make sure adding variables, constraints, and the objective all go as planned.

In [ ]:
### Print the model object to get a quick summary
print(model)

### Writing a *.lp file is a great way to get a look at your model
model.write('our_model.lp')

The `Slack` of a constraint is the gap between the left-hand side and right-hand side values at the optimal solution. You can also think of this as how far the value is from its bound, in our case upper bound. Printing this for the `resource_usage` constraints will show how much of each resource is remaining.

In [ ]:
for j in resources:
    print(f"Remaining {j}: {resource_usage[j].Slack}")

**Question**: Which one of the resource constraints is binding? Can a constraint be binding if the remaining resource value is not 0?

In [ ]:
## Let's remove the Mix Proportion constraints
model.remove([standard_charger_lb, fast_charger_ub, max_level2])
model.update()

## Look at which constraints remain in the model
model.getConstrs()

### Meet minimum service requirements

Management has some operational requirements:
- We must have **at least 5 fast chargers** (any combination of Fast50, Fast150, or Ultra350) for customers who need quick charging
- We must have **at least 15 standard chargers** (any combination of Level1 or Level2) for customers with more time

Recall that $F = \{Fast50, Fast150, Ultra350\}$ and $S = \{Level1, Level2\}$. We can model these minimum service requirements as:

\begin{align*}
\sum_{f \in F} x_f &\ge 5 \\
\sum_{s \in S} x_s &\ge 15
\end{align*}


In [ ]:
### Ensure we have diversity: at least 2 fast chargers and at least 3 standard chargers
min_fast = model.addConstr(gp.quicksum(x[f] for f in fast_chargers) >= 5, name="min_fast")
min_standard = model.addConstr(gp.quicksum(x[s] for s in standard_chargers) >= 15, name="min_standard")

solve_and_print_solution(model, x, chargers)

In [ ]:
### Print each charger type data here
chargers_data

### Cost of installing chargers:
- In the data frame above, we see there is an installation cost per charger. 
- Now management asks: **What's the cheapest way to serve at least 500 vehicles per day?**
- Write an expression for the total *variable cost* of chargers installed.
- Add a constraint that we must serve at least 500 vehicles per day.
- Make that the new objective to minimize total cost and solve.

Let $c_i$ be the installation cost of charger type $i$.
\begin{align*}
\text{total installation cost} = &\sum_i c_i*x_i \\
\text{Minimize} &\sum_i c_i*x_i \\
\text{subject to:} & \sum_i v_i*x_i \ge V_{min}
\end{align*}

In [ ]:
### Variable installation cost
vCost_all_chargers = sum(chargers_data.installation_cost[i] * x[i] for i in chargers)

### Add service level constraint - must serve at least 500 vehicles per day
min_service_level = 500
service_requirement = model.addConstr(vehicles_all_chargers >= min_service_level, name="service_requirement")

model.setObjective(vCost_all_chargers, GRB.MINIMIZE)

solve_and_print_solution(model, x, chargers)

print(f"\n=== Solution Analysis ===")
print(f"Total vehicles served: {vehicles_all_chargers.getValue():.0f}")
print(f"Total cost: ${vCost_all_chargers.getValue():,.0f}")
print(f"Cost per vehicle served: ${vCost_all_chargers.getValue() / vehicles_all_chargers.getValue():.2f}")

### Fixed costs: More decision variables
Our installation crew can only install one type of charger at a time. When we want to switch to a different type, we incur permitting and setup costs, which adds a fixed cost to each charger type. Given this, we have to decide whether or not we want to install each type of charger. If we decide to install at least one Fast50 charger, then we incur a fixed cost of $5,000.

**Key question**: Will adding fixed costs change which charger types we choose when trying to minimize cost while serving 500 vehicles/day?

Let's go step-by-step to add in the fixed costs for each charger type.
- First define a new set of variables indexed by charger type and call them `y`.
- Write a constraint that links the number of chargers installed with the new binary variable (**HINT:** use the cheat sheet).
- Don't resolve yet.

Let $f_i$ be the fixed cost of charger type $i$ and $y_i = 1$ if charger type $i$ is installed, $0$ otherwise.

If we decide to install a Fast50 charger, then $x_{Fast50} > 0$ and we want to have the associated cost go from 0 to the fixed cost.

So the constraint is
\begin{equation*}
x_i \le M*y_i, \forall i \in I
\end{equation*}

But what do we choose for $M$?

In [ ]:
# First, let's save the current solution for comparison
print("=== Current Solution (without fixed costs) ===")
current_cost = vCost_all_chargers.getValue()
current_vehicles = vehicles_all_chargers.getValue()
print(f"Total cost: ${current_cost:,.0f}")
print(f"Vehicles served: {current_vehicles:.0f}")
print("\nCharger types installed:")
for i in chargers:
    if x[i].X > 0.5:
        print(f"  {i}: {x[i].X:.0f} units (cost: ${chargers_data.installation_cost[i] * x[i].X:,.0f})")

y = model.addVars(chargers, vtype=GRB.BINARY, name="install")

### Link binary variable to charger installation using `big-M` constraints
M = 100
# If y=0, then x must be 0
link_x_y_upper = model.addConstrs((x[i] <= M * y[i] for i in chargers), name="link_upper")
# If x=0, then y must be 0 (equivalently: if x>0, then y must be 1)
link_x_y_lower = model.addConstrs((x[i] >= y[i] for i in chargers), name="link_lower")
model.update()


### Fixed costs: Combining costs
- Write an expression that finds the total fixed costs using the new variable and the `fixed_cost` column from `chargers_data`.
- Add that to the variable cost expression and set that as the new objective
- Solve!

**Question**: Do you think adding fixed costs will cause us to use fewer charger types?

\begin{align*}
&\text{total fixed cost} = \sum_i f_i*y_i \\
\text{total cost} = \space&\text{total fixed cost} + \text{total installation cost} = \sum_i f_i*y_i + c_i*x_i
\end{align*}

In [ ]:
fCost_all_chargers = gp.quicksum(chargers_data.fixed_cost[i] * y[i] for i in chargers)
tCost_all_chargers = vCost_all_chargers + fCost_all_chargers

model.setObjective(tCost_all_chargers)

solve_and_print_solution(model, x, chargers)

print(f"\n=== Solution Analysis ===")
print(f"Total vehicles served: {vehicles_all_chargers.getValue():.0f}")
print(f"Total cost: ${vCost_all_chargers.getValue():,.0f}")
print(f"Cost per vehicle served: ${vCost_all_chargers.getValue() / vehicles_all_chargers.getValue():.2f}")

### Logical constraints with binary variables
Now that we have binary decision variables that show which charger types are installed, we can model logical relationships between them. Model the following statements and write `gurobipy` code.

We won't solve a model with these.
- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).
- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).
- We install between 2 and 3 types of standard chargers.

- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).
\begin{equation*}
y_{Fast150} + y_{Ultra350} \ge 1
\end{equation*}

- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).
\begin{equation*}
y_{Fast150} + y_{Ultra350} = 1
\end{equation*}

- We install between 2 and 3 types of standard chargers.
\begin{equation*}
2 \le \sum_s y_s \le 3
\end{equation*}

In [ ]:
# We can install either Fast150 or Ultra350 chargers, and possibly both (at least one)
model.addConstr(y['Fast150'] + y['Ultra350'] >= 1, name="at_least_one_fast")

# We must install either Fast150 or Ultra350 chargers, but not both (exactly one)
model.addConstr(y['Fast150'] + y['Ultra350'] == 1, name="exactly_one_fast")

# We install between 2 and 3 types of standard chargers
model.addConstr(gp.quicksum(y[s] for s in standard_chargers) >= 2, name="min_standard_types")
model.addConstr(gp.quicksum(y[s] for s in standard_chargers) <= 3, name="max_standard_types")

### Another shortcut using the gurobipy API
model.addConstr(gp.quicksum(y[s] for s in standard_chargers) == [2,3], name="standard_types_range")

### Conditional statements
- If we install Level1 chargers, then we **must** install Level2 chargers.
- If we install Level1 chargers, then we **must** install Level2 **and** Fast50 chargers.
- If we install Level1 chargers, then we **must** install Level2 **or** Fast50 chargers (or both).

- If we install Level1 chargers, then we **must** install Level2 chargers.
\begin{align*}
y_{Level1} &\le y_{Level2}
\end{align*}

- If we install Level1 chargers, then we **must** install Level2 **and** Fast50 chargers.

\begin{align*}
y_{Level1} &\le y_{Level2}, \\
y_{Level1} &\le y_{Fast50} \\
&\text{or} \\
2*y_{Level1} &\le y_{Level2} + y_{Fast50}
\end{align*}

- If we install Level1 chargers, then we **must** install Level2 **or** Fast50 chargers (or both).

\begin{align*}
y_{Level1} &\le y_{Level2} + y_{Fast50}
\end{align*}

In [ ]:
#### Logical constraint candidates:

### If we install Level1 chargers, then we must install Level2 chargers
model.addConstr(y['Level1'] <= y['Level2'], name="level1_implies_level2")

### If we install Level1 chargers, then we must install Level2 and Fast50 chargers
model.addConstr(y['Level1'] <= y['Level2'], name="level1_implies_level2")
model.addConstr(y['Level1'] <= y['Fast50'], name="level1_implies_fast50")

model.addConstr(2*y['Level1'] <= y['Level2'] + y['Fast50'], name="level1_implies_both")

### If we install Level1 chargers, then we must install Level2 or Fast50 chargers
model.addConstr(y['Level1'] <= y['Level2'] + y['Fast50'], name="level1_implies_either")

## Multi-objective optimization
- We are asked to maximize the total vehicles served while minimizing costs.
- Math optimization cannot **simultaneously** work on two objectives -- there's always a tradeoff.
- Two types of multi-objective: hierarchical, and weighted (blended).
- After talking more about how we want to prioritize the objectives, we want to:
    - First maximize the **vehicles served per day**.
    - Then minimize total costs while not decreasing vehicles served by more than 10%
- Also include the following constraints we already discussed:
    - Resource limits
    - Meet minimum requirements
    - Make the most Level2 chargers

### DIY hierarchical multi-objective optimization
- Solve a model that maximizes the total vehicles served.
- Add a new constraint that uses this objective value as a new bound on total vehicles served. Let $z$ be this value.
\begin{equation*}
\sum_i v_i*x_i \ge 0.9*z
\end{equation*}
- Set the new objective to minimize total cost, as before.
- Solve!

In [ ]:
model = gp.Model("EV Charging Network")

M = 100

### Decision variables
x = model.addVars(chargers, vtype=GRB.INTEGER, name="chargers")
y = model.addVars(chargers, vtype=GRB.BINARY, name="install")

### Other expressions
vehicles_all_chargers = gp.quicksum(chargers_data.vehicles_per_day[i] * x[i] for i in chargers)
vCost_all_chargers = sum(chargers_data.installation_cost[i] * x[i] for i in chargers)
fCost_all_chargers = gp.quicksum(chargers_data.fixed_cost[i] * y[i] for i in chargers)
tCost_all_chargers = vCost_all_chargers + fCost_all_chargers

### Constraints
resource_usage = model.addConstrs((gp.quicksum(x[i] * requirements[i, j] for i in chargers) <= available[j] for j in resources), name="resource_usage")
max_level2 = model.addConstrs((x['Level2'] >= x[i] for i in [ii for ii in chargers if ii != 'Level2']), name='max_level2')
meet_min_fast = model.addConstr(gp.quicksum(x[f] for f in fast_chargers) >= 10, name="min_fast")
meet_min_standard = model.addConstr(gp.quicksum(x[s] for s in standard_chargers) >= 15, name="min_standard")
link_x_y_upper = model.addConstrs((x[i] <= M * y[i] for i in chargers), name="link_upper")
link_x_y_lower = model.addConstrs((x[i] >= y[i] for i in chargers), name="link_lower")

### First Objective
model.setObjective(vehicles_all_chargers, sense=GRB.MAXIMIZE)

solve_and_print_solution(model, x, chargers)

print(f"\n=== Solution Analysis ===")
print(f"Total vehicles served: {vehicles_all_chargers.getValue():.0f}")
print(f"Number of charger types installed: {sum(1 for i in chargers if y[i].X > 0.5)}")
print(f"Total cost: ${vCost_all_chargers.getValue():,.0f}")
total_cost_first_obj = vCost_all_chargers.getValue()


In [ ]:
z = model.ObjVal
print(z)
multi_obj = model.addConstr(vehicles_all_chargers >= 0.9*z)
model.setObjective(tCost_all_chargers, sense=GRB.MINIMIZE)
solve_and_print_solution(model, x, chargers)

print(f"\n=== Solution Analysis ===")
print(f"Minimum number of vehicles to be served: {0.9*z:.0f}")
print(f"Total vehicles served: {vehicles_all_chargers.getValue():.0f}")
print(f"Number of charger types installed: {sum(1 for i in chargers if y[i].X > 0.5)}")
print(f"Total cost: ${vCost_all_chargers.getValue():,.0f}")
total_cost_second_obj = vCost_all_chargers.getValue()
print(f"Cost savings using multi-objective: ${(total_cost_first_obj - total_cost_second_obj):,.0f}")

#### Homework!
Use the **cheat sheet** to do this using the `gurobipy` multi-objective functionality.

In [ ]:
model.dispose()